In [ ]:
# NOTE: Notebook này khớp với commit 1790d1d của repo AlexKeatg-FS.
# Sau khi clone, cell này sẽ in commit thực tế — nếu không khớp thì chạy: git -C /content/AlexKeatg-FS pull
import os
import shutil
import requests

REPO = 'AlexKeatg-FS'
BASE = '/content'
DEST = f'{BASE}/{REPO}'

# Lấy token: Colab Secrets (userdata) -> env -> nhập tay
TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not TOKEN:
    TOKEN = os.environ.get('GITHUB_TOKEN')
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (scope repo): ')

headers = {'Authorization': f'token {TOKEN}'}
resp = requests.get('https://api.github.com/user', headers=headers)
assert resp.status_code == 200, f'Token không hợp lệ: {resp.status_code} {resp.text[:200]}'
USER = resp.json()['login']
print('GitHub user:', USER)

if os.path.exists(DEST):
    shutil.rmtree(DEST)
!git clone --depth 1 https://{TOKEN}@github.com/{USER}/{REPO}.git {DEST}
assert os.path.exists(f'{DEST}/run.py'), 'Clone thất bại!'
!git -C {DEST} log --oneline -1
print('✅ Clone OK ->', DEST)

In [ ]:
%cd /content/AlexKeatg-FS
import os, subprocess
from pathlib import Path

MAMBA = Path('/content/micromamba/bin/micromamba')
MAMBA_ROOT = Path('/content/mamba_root')
ENV = MAMBA_ROOT / 'envs' / 'alexkeatg310'
MAIN_PY = ENV / 'bin' / 'python'

def run(cmd):
    cmd = [str(x) for x in cmd]
    print('$', ' '.join(cmd), flush=True)
    p = subprocess.run(cmd, check=False)
    if p.returncode != 0:
        raise RuntimeError('Command failed: ' + ' '.join(cmd))
    return p

# 1) Micromamba (chỉ cài lần đầu)
if not MAMBA.exists():
    run(['bash', '-lc',
         'mkdir -p /content/micromamba && '
         'curl -L --fail --retry 5 -s https://micro.mamba.pm/api/micromamba/linux-64/latest '
         '| tar -xj -C /content/micromamba'])
assert MAMBA.exists(), MAMBA

# 2) Env python 3.10 (chỉ tạo lần đầu)
if not MAIN_PY.exists():
    e = os.environ.copy(); e['MAMBA_ROOT_PREFIX'] = str(MAMBA_ROOT)
    subprocess.run([str(MAMBA), 'create', '-y', '-n', 'alexkeatg310', '-c', 'conda-forge', 'python=3.10', 'pip'], env=e, check=True)

# 3) Cài packages vào ENV RIÊNG -> không đụng kernel Colab -> KHÔNG nhắc restart
# Gỡ torch nếu env cũ còn (đã bỏ hẳn — mọi thứ chạy ONNX)
run([MAIN_PY, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision'])
run([MAIN_PY, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run([MAIN_PY, '-m', 'pip', 'install', '-q', 'numpy<2.0', 'opencv-python-headless', 'onnx', 'insightface',
     'scipy', 'scikit-image', 'psutil', 'tqdm', 'requests', 'fastapi', 'gradio==5.13.0'])
run([MAIN_PY, '-m', 'pip', 'install', '-q', '--force-reinstall', 'pydantic==2.10.6'])
run([MAIN_PY, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'])
run([MAIN_PY, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu'])  # bản mới nhất (không cần pin)

# 4) Verify CUDA trong env
run([MAIN_PY, '-c',
     'import onnxruntime as ort; '
     'print("ORT providers:", ort.get_available_providers())'])
print('✅ Env OK — KHÔNG cần restart runtime. Chạy cell 3.')

In [ ]:
%cd /content/AlexKeatg-FS
!rm -f config.yaml  # luôn dùng defaults mới (threads 8, crf 22, DFL XSeg, erosion 2)
import os
PY = '/content/mamba_root/envs/alexkeatg310/bin'
assert os.path.exists(PY + '/python'), 'Chạy cell 2 trước!'
os.environ['PATH'] = PY + ':' + os.environ.get('PATH', '')
os.environ['MPLBACKEND'] = 'Agg'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'

!{PY}/python -u run.py